# 🔑 Solutions: Lesson 06 · Optimizers, Learning Rates, Epochs & Batch Sizes

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 6.1: Count the updates  &nbsp; `⭐`

Write `training_steps(n_samples, batch_size, epochs)` returning `(steps_per_epoch, total_steps)`. The final partial batch **counts** as a step (PyTorch's default `drop_last=False`).


In [2]:
import math

def training_steps(n_samples, batch_size, epochs):
    spe = math.ceil(n_samples / batch_size)
    return spe, spe * epochs

In [3]:
# ▶ self-check
dl.check("MNIST, B=128, 10 epochs → (469, 4690)", training_steps(60000, 128, 10) == (469, 4690))
dl.check("exact division: 1000/100, 3 epochs → (10, 30)", training_steps(1000, 100, 3) == (10, 30))
dl.check("batch bigger than data → 1 step per epoch", training_steps(50, 64, 4) == (1, 4))

✅ MNIST, B=128, 10 epochs → (469, 4690)
✅ exact division: 1000/100, 3 epochs → (10, 30)
✅ batch bigger than data → 1 step per epoch


---
### 🧩 Problem 6.2: Momentum from scratch  &nbsp; `⭐⭐`

Implement `momentum_descent(grad, p0, lr, beta, steps)` using the *velocity* form
$v\leftarrow\beta v+g,\ \theta\leftarrow\theta-\eta v$ (start with $v=0$). `p0` and the gradient are NumPy arrays. Return the final parameters.


In [4]:
def momentum_descent(grad, p0, lr, beta, steps):
    p, v = p0.astype(float), np.zeros_like(p0, dtype=float)
    for _ in range(steps):
        v = beta * v + grad(p)
        p = p - lr * v
    return p

In [5]:
# ▶ self-check
A = np.array([1.0, 100.0]); g = lambda p: A * p
one = momentum_descent(g, np.array([1.0, 1.0]), 0.01, 0.9, 1)
dl.check("one step equals plain SGD (v starts at 0): [0.99, 0.0]", one is not None and np.allclose(one, [0.99, 0.0]))
two = momentum_descent(g, np.array([1.0, 1.0]), 0.01, 0.9, 2)
dl.check("two steps: x=0.99−0.01(0.9·1+0.99)=0.9711", two is not None and np.isclose(two[0], 0.9711))
res = momentum_descent(g, np.array([-4.0, 1.0]), 0.004, 0.9, 200)
dl.check("converges on the ravine", res is not None and np.linalg.norm(res) < 0.05)

✅ one step equals plain SGD (v starts at 0): [0.99, 0.0]
✅ two steps: x=0.99−0.01(0.9·1+0.99)=0.9711
✅ converges on the ravine


---
### 🧩 Problem 6.3: Adam from scratch (matches PyTorch!)  &nbsp; `⭐⭐⭐`

Implement `adam_run(grad, p0, lr, steps)` with $\beta_1=0.9,\ \beta_2=0.999,\ \epsilon=10^{-8}$ and bias correction (steps are numbered $t=1,2,\dots$).
It should reproduce `torch.optim.Adam` to 5 decimal places.


In [6]:
def adam_run(grad, p0, lr, steps, b1=0.9, b2=0.999, eps=1e-8):
    p, m, v = p0.astype(float), np.zeros_like(p0, dtype=float), np.zeros_like(p0, dtype=float)
    for t in range(1, steps + 1):
        g = grad(p)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        p = p - lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps)
    return p

In [7]:
# ▶ self-check
import torch
A = np.array([1.0, 10.0, 3.0]); p0 = np.array([2.0, -1.0, 0.5])
mine = adam_run(lambda p: A * p, p0, 0.1, 25)
th = torch.tensor(p0, requires_grad=True)
opt = torch.optim.Adam([th], lr=0.1)
for _ in range(25):
    opt.zero_grad(); (0.5 * (torch.tensor(A) * th ** 2).sum()).backward(); opt.step()
dl.check("matches torch.optim.Adam after 25 steps", mine is not None and np.allclose(mine, th.detach().numpy(), atol=1e-5))

✅ matches torch.optim.Adam after 25 steps


**Why this works.** Adam is just ~6 lines. Frameworks add speed, parameter groups and numerical care: not new maths.

---
### 🧩 Problem 6.4: Run your own learning-rate sweep  &nbsp; `⭐⭐`

Complete `sweep(lrs)`: for each learning rate, build a fresh model with `make_model()`, train it with **Adam** for 3 epochs on `tr_s` using `dl.fit(..., verbose=False)`,
and record the **final training loss**. Return the list of losses. Then pick the best learning rate (lowest loss) into `best_lr`.
(Data loaders are provided.)


In [8]:
def make_model(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 10))
cpu = torch.device("cpu")
xtr, ytr, xva, yva = dl.load_mnist(n_train=2000, n_test=1000, flatten=True)
tr_s = DataLoader(TensorDataset(torch.tensor(xtr), torch.tensor(ytr)), batch_size=64, shuffle=True)
va_s = DataLoader(TensorDataset(torch.tensor(xva), torch.tensor(yva)), batch_size=512)

lrs = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
def sweep(lrs):
    out = []
    for lr in lrs:
        model = make_model()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        h = dl.fit(model, tr_s, va_s, nn.CrossEntropyLoss(), opt, epochs=3, device=cpu, verbose=False)
        out.append(h["train_loss"][-1])
    return out

losses = sweep(lrs)
best_lr = lrs[int(np.argmin(losses))]

In [9]:
# ▶ self-check
ok = losses is not None and len(losses) == len(lrs)
dl.check("one loss per learning rate", ok)
dl.check("lr=1e-5 (too low) is worse than lr=1e-3", ok and losses[0] > losses[2])
dl.check("lr=1.0 (too high for Adam) is worse than lr=1e-3", ok and losses[-1] > losses[2])
dl.check("best_lr is between 1e-4 and 1e-1", best_lr is not None and 1e-4 <= best_lr <= 1e-1)

✅ one loss per learning rate
✅ lr=1e-5 (too low) is worse than lr=1e-3
✅ lr=1.0 (too high for Adam) is worse than lr=1e-3
✅ best_lr is between 1e-4 and 1e-1


**Why this works.** A log-spaced sweep reveals the U-shape immediately. Always check *both* ends: if the best value is at the edge of your range, extend the range.